## Causality Anlysis 

In [2]:
!pip install statsmodels

## Data Prepared

In [44]:
import pandas as pd

# 1. Load wildfire yang sudah punya kolom province (yang barusan kamu bikin)
wildfire_with_province = pd.read_csv(r'C:\Users\hanif\Downloads\Data Analyst\Project\end-to-end\Indonesia Deforestation & Wildfire Analysis (2001–2024)\indonesia-deforestation-wildfire-analysis\data\processed\wildfire_with_province.csv')

# filter dulu
wildfire_clean = wildfire_with_province[
    (wildfire_with_province['confidence'] > 50) &
    (wildfire_with_province['province'].notna())
].copy()

wildfire_clean['year'] = pd.to_datetime(wildfire_clean['acq_date']).dt.year

wildfire_clean.head()

,latitude,longitude,acq_date,acq_time,satellite,brightness,bright_t31,confidence,frp,daynight,year,month,province
0,0.8381,101.5897,2000-11-01,329,Terra,346.1,293.5,94,186.0,D,2000,11,Riau
1,0.8349,101.6121,2000-11-01,329,Terra,335.3,294.2,88,115.5,D,2000,11,Riau
2,0.8455,101.5838,2000-11-01,329,Terra,324.0,293.8,80,55.8,D,2000,11,Riau
3,0.8421,101.6062,2000-11-01,329,Terra,332.7,293.9,86,96.7,D,2000,11,Riau
4,-0.2439,101.5847,2000-11-01,329,Terra,329.6,293.6,83,74.9,D,2000,11,Riau


In [46]:
# 2. Agregasi ke provinsi-tahun
hotspot_panel = wildfire_clean.groupby(['province', 'year']).agg(
    hotspot_count=('year', 'size'),
    mean_frp=('frp', 'mean'),
    total_frp=('frp', 'sum')
).reset_index()

In [48]:
# 3. Load data GFW
gfw = pd.read_csv(r'C:\Users\hanif\Downloads\Data Analyst\Project\end-to-end\Indonesia Deforestation & Wildfire Analysis (2001–2024)\indonesia-deforestation-wildfire-analysis\data\processed\subnational_1_tree_cover_loss.csv')
gfw = gfw.copy()

gfw_long = gfw.rename(columns={
    'subnational1': 'province',
    'tree_cover_loss_ha': 'tc_loss_ha'
})[['province', 'year', 'tc_loss_ha']]

In [50]:
gfw_long = gfw_long[gfw_long['year'] <= 2024]

merged_panel = gfw_long.merge(hotspot_panel, on=['province', 'year'], how='left')
merged_panel['hotspot_count'] = merged_panel['hotspot_count'].fillna(0)
merged_panel['total_frp'] = merged_panel['total_frp'].fillna(0)

print(merged_panel['year'].min(), merged_panel['year'].max())
print(merged_panel.shape)

2001 2024
(816, 6)


In [37]:
merged_panel.head()

,province,year,tc_loss_ha,hotspot_count,mean_frp,total_frp
0,Aceh,2001,18278,78.0,39.491026,3080.3
1,Bali,2001,424,0.0,NaN,0.0
2,Bangka Belitung,2001,13910,11.0,24.963636,274.6
3,Banten,2001,1054,11.0,47.000000,517.0
4,Bengkulu,2001,14198,33.0,19.318182,637.5


## Contemporary Correlation

In [39]:
from scipy.stats import pearsonr, spearmanr

r, p = pearsonr(merged_panel['hotspot_count'], merged_panel['tc_loss_ha'])
rs, ps = spearmanr(merged_panel['hotspot_count'], merged_panel['tc_loss_ha'])

print(f"Pearson  r={r:.3f}, p={p:.2e}")
print(f"Spearman r={rs:.3f}, p={ps:.2e}")

Pearson  r=0.621, p=4.08e-88
Spearman r=0.727, p=4.83e-135


## Time-lag correlation

In [41]:
merged_panel = merged_panel.sort_values(['province', 'year'])
lag_rows = []

for prov, g in merged_panel.groupby('province'):
    g = g.sort_values('year').reset_index(drop=True)
    if len(g) > 1:
        a = g['hotspot_count'][:-1].values
        b = g['tc_loss_ha'][1:].values
        lag_rows.append(pd.DataFrame({'hotspot_t': a, 'tcloss_t1': b}))

lagdf = pd.concat(lag_rows, ignore_index=True)
r_lag, p_lag = pearsonr(lagdf['hotspot_t'], lagdf['tcloss_t1'])
print(f"Hotspot(t) -> TC_loss(t+1): r={r_lag:.3f}, p={p_lag:.2e}")

Hotspot(t) -> TC_loss(t+1): r=0.664, p=9.91e-101


In [43]:
lag_rows2 = []
for prov, g in merged_panel.groupby('province'):
    g = g.sort_values('year').reset_index(drop=True)
    if len(g) > 1:
        a = g['tc_loss_ha'][:-1].values
        b = g['hotspot_count'][1:].values
        lag_rows2.append(pd.DataFrame({'tcloss_t': a, 'hotspot_t1': b}))

lagdf2 = pd.concat(lag_rows2, ignore_index=True)
r_lag2, p_lag2 = pearsonr(lagdf2['tcloss_t'], lagdf2['hotspot_t1'])
print(f"TC_loss(t) -> Hotspot(t+1): r={r_lag2:.3f}, p={p_lag2:.2e}")

TC_loss(t) -> Hotspot(t+1): r=0.513, p=8.23e-54


## Percentage Wildfire per Province

In [12]:
import pandas as pd

df = pd.read_csv(r'C:\Users\hanif\Downloads\Data Analyst\Project\end-to-end\Indonesia Deforestation & Wildfire Analysis (2001–2024)\indonesia-deforestation-wildfire-analysis\data\processed\subnational1_drivers.csv')

# total loss per provinsi, SEMUA driver dijumlah, SEMUA tahun dijumlah
total_per_prov = df.groupby('subnational1')['tree_cover_loss_ha'].sum()

wildfire_only = df[df['driver']=='Wildfire'].groupby('subnational1')['tree_cover_loss_ha'].sum()

# reindex supaya semua 34 provinsi ada, isi 0 kalau tidak ada baris wildfire
wildfire_only = wildfire_only.reindex(total_per_prov.index, fill_value=0)

pct_wildfire = (wildfire_only / total_per_prov * 100).round(2)
print(pct_wildfire.sort_values(ascending=False))

subnational1
Kepulauan Riau         29.92
Kalimantan Tengah      23.26
Maluku                 18.21
Maluku Utara           11.37
Sumatera Selatan        9.03
Sulawesi Tengah         7.92
Papua Barat             6.94
Papua                   6.88
Kalimantan Barat        5.50
Kalimantan Selatan      4.98
Riau                    4.55
Jambi                   4.34
Sulawesi Selatan        4.05
Bangka Belitung         3.62
Kalimantan Timur        3.39
Sulawesi Utara          2.81
Sulawesi Tenggara       2.75
Kalimantan Utara        1.79
Lampung                 1.51
Nusa Tenggara Timur     1.25
Nusa Tenggara Barat     1.06
Sumatera Utara          0.75
Gorontalo               0.73
Jawa Timur              0.69
Sumatera Barat          0.61
Aceh                    0.48
Sulawesi Barat          0.43
Bali                    0.42
Banten                  0.36
Jawa Barat              0.32
Jawa Tengah             0.07
Bengkulu                0.06
Jakarta Raya            0.00
Yogyakarta              0.00
N

## Driver Breakdown

In [38]:
pivot = df.groupby(['subnational1', 'driver'])['tree_cover_loss_ha'].sum().unstack(fill_value=0)
pivot['total'] = pivot.sum(axis=1)

driver_cols = [c for c in pivot.columns if c != 'total']

pivot['dominant_driver'] = pivot[driver_cols].idxmax(axis=1)
pivot['dominant_pct'] = (pivot[driver_cols].max(axis=1) / pivot['total'] * 100).round(1)

result = pivot[['dominant_driver', 'dominant_pct', 'total']].reset_index()
result = result.rename(columns={'subnational1': 'province', 'total': 'tc_loss_ha'})

result = result.sort_values('tc_loss_ha', ascending=False)

pd.set_option('display.max_rows', 40)

In [40]:
result

driver,province,dominant_driver,dominant_pct,tc_loss_ha
24,Riau,Permanent agriculture,74.4,4397585
11,Kalimantan Barat,Permanent agriculture,76.6,4354938
13,Kalimantan Tengah,Permanent agriculture,56.8,3959340
31,Sumatera Selatan,Permanent agriculture,77.2,3404617
14,Kalimantan Timur,Permanent agriculture,71.9,3252700
7,Jambi,Permanent agriculture,79.0,2081755
32,Sumatera Utara,Permanent agriculture,84.1,1693948
12,Kalimantan Selatan,Permanent agriculture,75.6,959789
0,Aceh,Permanent agriculture,85.8,915248
27,Sulawesi Tengah,Permanent agriculture,53.6,874411


In [36]:
# gabungkan driver breakdown (result) dengan wildfire percentage (pct_wildfire)
combined = result.merge(
    pct_wildfire.rename('wildfire_pct'),
    left_on='province', right_index=True, how='left'
)

combined = combined[['province', 'dominant_driver', 'dominant_pct', 'wildfire_pct', 'tc_loss_ha']]
combined = combined.sort_values('tc_loss_ha', ascending=False).reset_index(drop=True)

print(combined.to_string(index=False))
combined.to_csv(r'C:\Users\hanif\Downloads\Data Analyst\Project\end-to-end\Indonesia Deforestation & Wildfire Analysis (2001–2024)\indonesia-deforestation-wildfire-analysis\data\processed\driver_and_wildfire_summary.csv', index=False)

           province              dominant_driver  dominant_pct  wildfire_pct  tc_loss_ha
               Riau        Permanent agriculture          74.4          4.55     4397585
   Kalimantan Barat        Permanent agriculture          76.6          5.50     4354938
  Kalimantan Tengah        Permanent agriculture          56.8         23.26     3959340
   Sumatera Selatan        Permanent agriculture          77.2          9.03     3404617
   Kalimantan Timur        Permanent agriculture          71.9          3.39     3252700
              Jambi        Permanent agriculture          79.0          4.34     2081755
     Sumatera Utara        Permanent agriculture          84.1          0.75     1693948
 Kalimantan Selatan        Permanent agriculture          75.6          4.98      959789
               Aceh        Permanent agriculture          85.8          0.48      915248
    Sulawesi Tengah        Permanent agriculture          53.6          7.92      874411
   Kalimantan Utara  